# Retrievers in RAG — Complete Class Notes (Groq Version)

**Paper used for practice:** Llama 2 research paper (77 pages)

**What changed from the class notebook**
- **LLM:** OpenAI `ChatOpenAI` → **Groq `ChatGroq`** (model: `openai/gpt-oss-120b`)
- **Embeddings:** OpenAI embeddings → **free local HuggingFace embeddings**, because Groq does **not** provide an embedding model.

**Topics covered (in order)**
1. Setup (install, imports, API key)
2. Loading the PDF
3. Adding custom metadata
4. Chunking
5. Embeddings
6. Vector store (Chroma)
7. What is a Retriever?
8. Similarity search retriever
9. MMR retriever
10. Relevance scores & score-threshold retriever
11. Distance metrics (cosine, euclidean, dot product)
12. Metadata filtering
13. Pre-filtering vs Post-filtering
14. Sparse retrieval (BM25)
15. Dense retrieval
16. Hybrid retrieval (BM25 + Dense + RRF)
17. Groq LLM setup
18. Query rewriting
19. Query expansion
20. MultiQueryRetriever (built-in)
21. Query decomposition
22. HyDE (custom retriever)
23. HyDE (LangChain built-in)
24. Reranking with a Cross-Encoder
25. Hybrid retrieval + Reranking (best pipeline)
26. Summary
27. Homework topics

---
## Topic 1: Setup

### 1.1 Install packages
Run this cell once. After it finishes, restart the kernel.

| Package | Why we need it |
|---|---|
| `langchain-groq` | Groq chat model (`ChatGroq`) |
| `langchain-huggingface` + `sentence-transformers` | Free local embeddings and the cross-encoder reranker |
| `langchain-chroma` | Chroma vector database |
| `langchain-community` | `PyPDFLoader`, `BM25Retriever`, `HuggingFaceCrossEncoder` |
| `langchain-classic` | `EnsembleRetriever`, `MultiQueryRetriever`, `ContextualCompressionRetriever`, HyDE |
| `pypdf` | Reads PDF files |
| `rank_bm25` | The BM25 keyword-search algorithm |

In [ ]:
%pip install -U langchain langchain-classic langchain-community langchain-groq langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers pypdf rank_bm25 pandas numpy

### 1.2 Imports
All imports are kept in one place so the notebook runs from top to bottom without errors.

In [ ]:
import getpass
import os
import shutil
from pathlib import Path
from typing import Any, List

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

# Loading and chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings and vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LLM (Groq)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Retrievers
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print("All imports loaded.")

### 1.3 Groq API key
Get a free key from **console.groq.com → API Keys**. The cell asks for the key only if it is not already set.

> Tip: `langchain-community` may show a *DeprecationWarning*. It is only a warning; the code still works.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

print("Groq API key configured.")

---
## Topic 2: Loading the PDF

**What:** A *document loader* reads a file and turns it into LangChain `Document` objects.

**How `PyPDFLoader` works:** it creates **one `Document` per PDF page**. Each `Document` has:
- `page_content` → the text of that page
- `metadata` → extra info such as `source` (file path) and `page` (page index, starting from 0)

The cell below first looks for `llama2-research-paper.pdf`. If the name is slightly different, it picks the only PDF in the folder automatically.

In [ ]:
DATA_DIR = Path(
    r"D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0"
    r"\Class-36-29-July-2026-Retriever\data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(f"No PDF file was found inside:\n{DATA_DIR}")
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:", PDF_PATH)

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

print("\nFirst-page metadata:")
print(pages[0].metadata)

print("\nFirst 500 characters:")
print(pages[0].page_content[:500])

---
## Topic 3: Adding Custom Metadata

**What:** We attach our own labels (metadata) to every page, e.g. which **section** of the paper it belongs to.

**Why it matters:**
1. **Filtering** — later we can search *only* inside the `fine_tuning` section.
2. **Citations** — we can show the user the page and section an answer came from.
3. **Access control** — a field like `access_level` can hide private documents.

**How:** We use the page number to decide the section (e.g. pages 8–19 are fine-tuning). `PyPDFLoader` counts pages from 0, so we add 1 to get the real page number.

In [ ]:
def identify_section(paper_page: int) -> str:
    # Return the section name of the Llama 2 paper for a given page number.
    if 1 <= paper_page <= 2:
        return "front_matter"
    if 3 <= paper_page <= 4:
        return "introduction"
    if 5 <= paper_page <= 7:
        return "pretraining"
    if 8 <= paper_page <= 19:
        return "fine_tuning"
    if 20 <= paper_page <= 31:
        return "safety"
    if 32 <= paper_page <= 35:
        return "discussion"
    if paper_page == 36:
        return "conclusion"
    if 37 <= paper_page <= 45:
        return "references"
    if 46 <= paper_page <= 77:
        return "appendix"
    return "unknown"


for page_document in pages:
    paper_page = int(page_document.metadata.get("page", 0)) + 1  # 0-based -> real page number

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

for page_document in pages[:5]:
    print(page_document.metadata["paper_page"], page_document.metadata["section"])

---
## Topic 4: Chunking (Splitting Text)

**What:** We cut each page into smaller pieces called **chunks**.

**Why:** One page is too big. A small chunk contains one focused idea, so search becomes more accurate, and we only send relevant text to the LLM.

**`RecursiveCharacterTextSplitter` settings:**
| Setting | Meaning |
|---|---|
| `chunk_size=1000` | Each chunk is at most ~1000 characters |
| `chunk_overlap=200` | Neighbouring chunks share 200 characters, so a sentence cut at the edge is not lost |
| `add_start_index=True` | Saves where the chunk starts inside the page (`start_index`) |

"Recursive" means it first tries to split on paragraphs (`\n\n`), then lines (`\n`), then spaces, so it keeps sentences together as much as possible.

Every chunk **keeps the metadata of its page**, and we also give each chunk a unique `chunk_id`.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")
    chunk.metadata["chunk_id"] = f"llama2-page-{paper_page}-chunk-{chunk_number}"

print(f"Total pages:  {len(pages)}")
print(f"Total chunks: {len(chunks)}")
print("\nExample chunk metadata:")
print(chunks[0].metadata)

---
## Topic 5: Embeddings

**What:** An embedding model turns text into a list of numbers (a **vector**). Texts with similar *meaning* get vectors that are close to each other.

**Why HuggingFace and not Groq?** Groq only offers chat/LLM models, **no embedding model**. So we use a free model that runs on your own computer:
- `sentence-transformers/all-MiniLM-L6-v2` → small, fast, gives **384** numbers per text.
- The first run downloads the model (~90 MB) once.

`normalize_embeddings=True` makes every vector length = 1. This makes cosine similarity and dot product give the same result (we prove this in Topic 11).

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

test_vector = embeddings.embed_query("What is Llama 2?")

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

---
## Topic 6: Vector Store (Chroma)

**What:** A vector store is a database that saves each chunk together with its embedding, and can quickly find the chunks closest to a query vector.

**Key settings:**
- `persist_directory` → saves the database on disk, so we don't have to embed again every time.
- `collection_name` → name of the table inside Chroma.
- `"hnsw": {"space": "cosine"}` → use **cosine** distance. HNSW is the fast search index Chroma uses.

**`REBUILD_INDEX` flag:**
- `True` → delete the old database and build a new one.
- `False` → just load the saved database (faster).

> **Important:** a database built with one embedding model can't be searched with another (OpenAI vectors had 1536 numbers, MiniLM has 384). That's why this version uses a **new folder name** (`chroma_llama2_retriever_hf`). Always keep the same embedding model for building and searching.

In [ ]:
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever_hf"
COLLECTION_NAME = "llama2_retriever_demo_hf"

# Set to False after the first run to reuse the saved index.
REBUILD_INDEX = True

if REBUILD_INDEX or not PERSIST_DIRECTORY.exists():
    shutil.rmtree(PERSIST_DIRECTORY, ignore_errors=True)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(PERSIST_DIRECTORY),
        collection_configuration={"hnsw": {"space": "cosine"}},
    )
    print("New vector store created.")
else:
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(PERSIST_DIRECTORY),
    )
    print("Existing vector store loaded.")

print(f"Stored chunks: {len(chunks)}")
print(f"Saved at: {PERSIST_DIRECTORY}")

---
## Topic 7: What is a Retriever?

**Vector store vs Retriever:**
- A **vector store** stores and searches vectors.
- A **retriever** is a simple interface: *give it a question → it returns a list of relevant `Document`s*. You call it with `.invoke(query)`.

Any vector store can become a retriever with `vector_store.as_retriever(...)`. Because every retriever has the same `.invoke()` interface, you can swap search strategies without changing the rest of your RAG code.

**The 3 `search_type` options:**
| `search_type` | Idea |
|---|---|
| `"similarity"` | Return the top-k most similar chunks |
| `"mmr"` | Return relevant **and** diverse chunks |
| `"similarity_score_threshold"` | Return only chunks above a minimum score |

Below is a helper function to print results neatly.

In [ ]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 500,
) -> None:
    # Print retrieved LangChain Documents in a readable way.
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents[:max_documents], start=1):
        metadata = document.metadata
        print(f"\nRANK: {rank}")
        print(f"Page: {metadata.get('paper_page')} | Section: {metadata.get('section')} | Chunk: {metadata.get('chunk_id')}")
        print("-" * 90)
        print(document.page_content[:max_characters])


def show_ids(documents, title: str) -> None:
    # Short view: only page, section and chunk id.
    print(f"\n{title}")
    for rank, document in enumerate(documents, start=1):
        print(rank, document.metadata.get("paper_page"), document.metadata.get("section"), document.metadata.get("chunk_id"))

---
## Topic 8: Similarity Search Retriever

**What:** The basic retriever. It converts the query into a vector and returns the **k** closest chunks.

- `k=4` → return 4 chunks.

**Weakness:** the top results can be **near-duplicates** (e.g. 4 chunks saying the same thing), so you waste context space.

In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

query = "What model sizes of Llama 2 were released?"
similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents, title="Similarity Search Results")

---
## Topic 9: MMR Retriever (Maximal Marginal Relevance)

**Problem it solves:** similarity search may return repeated information.

**How MMR works:**
1. First fetch a bigger pool of candidates (`fetch_k=20`).
2. Then pick `k=4` one by one. Each new pick must be **relevant to the query** but **different from the chunks already picked**.

**`lambda_mult` (0 to 1):**
| Value | Behaviour |
|---|---|
| close to **1** | more relevance (behaves like normal similarity) |
| close to **0** | more diversity |
| **0.5** | balance of both |

**Use MMR when** you want wide coverage of a topic (e.g. "How was Llama 2-Chat trained?" touches pretraining, SFT, RLHF...).

In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    },
)

query = "How was Llama 2-Chat trained and aligned?"
mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents, title="MMR Results")

### Similarity vs MMR — side by side
Look at the sections: similarity results usually come from the same area, MMR results are spread across more sections.

In [ ]:
query = "How was Llama 2-Chat trained and aligned?"

show_ids(similarity_retriever.invoke(query), "Similarity Search results:")
show_ids(mmr_retriever.invoke(query), "MMR results:")

---
## Topic 10: Relevance Scores & Score-Threshold Retriever

**Relevance score:** a number between **0 and 1** telling how relevant a chunk is (higher = better). With cosine space, `relevance = 1 − cosine distance`.

**Step 1 — look at the scores first** with `similarity_search_with_relevance_scores()`. This tells you what a "good" score looks like for your embedding model.

**Step 2 — score-threshold retriever:** returns only chunks whose score is **above** `score_threshold` (max `k` chunks). If nothing is good enough, it returns **nothing**, which is useful to avoid sending junk to the LLM.

> The right threshold depends on the embedding model. MiniLM scores are different from OpenAI scores, so check Step 1 and adjust the threshold.

In [ ]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(query=query, k=5)

for rank, (document, relevance_score) in enumerate(scored_results, start=1):
    print("=" * 90)
    print(f"Rank: {rank} | Relevance score: {relevance_score:.4f} | Page: {document.metadata.get('paper_page')} | Section: {document.metadata.get('section')}")
    print(document.page_content[:300])

In [ ]:
# Adjust this value after looking at the scores above.
SCORE_THRESHOLD = 0.5

threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": SCORE_THRESHOLD,
    },
)

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents, title=f"Score-Threshold Results (score >= {SCORE_THRESHOLD})")

---
## Topic 11: Distance Metrics (Cosine, Euclidean, Dot Product)

These are the 3 common ways to measure how close two vectors are:

| Metric | Formula idea | Better when | Looks at |
|---|---|---|---|
| **Cosine similarity** | angle between vectors | **higher** | direction only (meaning) |
| **Euclidean distance** | straight-line distance | **lower** | direction + length |
| **Dot product** | multiply & add | **higher** | direction + length |

**Key fact:** when vectors are **normalized** (length = 1):
- dot product = cosine similarity
- euclidean distance gives the **same ranking** as cosine

So for normalized embeddings, all three metrics rank the chunks in the same order. We will prove it below.

In [ ]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(metric_query, k=6)
candidate_texts = [document.page_content for document in candidate_documents]

query_vector = np.asarray(embeddings.embed_query(metric_query), dtype=np.float64)
document_vectors = np.asarray(embeddings.embed_documents(candidate_texts), dtype=np.float64)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

In [ ]:
def cosine_similarity(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)


def euclidean_distance(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(np.linalg.norm(vector_a - vector_b))


def dot_product(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(np.dot(vector_a, vector_b))


metric_rows = []

for document, document_vector in zip(candidate_documents, document_vectors):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "cosine_similarity": cosine_similarity(query_vector, document_vector),
            "euclidean_distance": euclidean_distance(query_vector, document_vector),
            "dot_product": dot_product(query_vector, document_vector),
            "preview": document.page_content[:80].replace("\n", " "),
        }
    )

metric_table = pd.DataFrame(metric_rows)
metric_table

### Ranking by each metric
Cosine and dot product are sorted **high → low**, euclidean is sorted **low → high**. The page order comes out the same in all three.

In [ ]:
print("By cosine (high first):   ", metric_table.sort_values("cosine_similarity", ascending=False)["paper_page"].tolist())
print("By euclidean (low first): ", metric_table.sort_values("euclidean_distance", ascending=True)["paper_page"].tolist())
print("By dot product (high first):", metric_table.sort_values("dot_product", ascending=False)["paper_page"].tolist())

### Proof: normalized dot product = cosine similarity

In [ ]:
normalized_query_vector = query_vector / np.linalg.norm(query_vector)
normalized_document_vectors = document_vectors / np.linalg.norm(document_vectors, axis=1, keepdims=True)

normalized_dot_scores = normalized_document_vectors @ normalized_query_vector
cosine_scores = np.asarray([cosine_similarity(query_vector, v) for v in document_vectors])

print("Cosine scores:              ", cosine_scores)
print("Dot product after normalize:", normalized_dot_scores)
print("Approximately equal?", np.allclose(cosine_scores, normalized_dot_scores, atol=1e-8))

---
## Topic 12: Metadata Filtering

**What:** Search **only** among chunks whose metadata matches a condition. This is where the metadata from Topic 3 pays off.

**Simple filter:** `{"section": "fine_tuning"}` → only fine-tuning chunks.

**Chroma filter operators:**
| Operator | Meaning |
|---|---|
| `$eq` / `$ne` | equal / not equal |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than |
| `$in` / `$nin` | value is in / not in a list |
| `$and` / `$or` | combine several conditions |

> With more than one condition, Chroma needs an explicit `$and` (or `$or`).

In [ ]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"section": "fine_tuning"},
    },
)

fine_tuning_documents = fine_tuning_retriever.invoke("How was Llama 2-Chat aligned with human preferences?")
display_documents(fine_tuning_documents, title="Filter: section = fine_tuning")

# Check: every result must be from fine_tuning
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"
print("\nAll returned documents are from the fine_tuning section.")

### Multiple conditions with `$and`

In [ ]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {"section": {"$eq": "fine_tuning"}},
                {"year": {"$eq": 2023}},
                {"organization": {"$eq": "Meta"}},
            ]
        },
    },
)

filtered_documents = filtered_retriever.invoke("How was human preference data collected?")
display_documents(filtered_documents, title="Filter: fine_tuning AND 2023 AND Meta")

---
## Topic 13: Pre-filtering vs Post-filtering

| | **Pre-filtering** | **Post-filtering** |
|---|---|---|
| When is the filter applied? | **Before** the vector search | **After** the vector search |
| How | the database only searches matching chunks | search everything, then remove non-matching results in Python |
| Number of results | always gets `k` matching results (if they exist) | may end up with **fewer** than you wanted |
| Speed | faster on big datasets | wastes work on chunks that get thrown away |
| When to use | filter is known in advance (section, user, date) | filter logic is complex and the database can't express it |

The `filter` we passed to Chroma above **is pre-filtering**.

In [ ]:
# PRE-FILTERING: filter goes into the search itself
pre_filter = {
    "$and": [
        {"section": {"$eq": "fine_tuning"}},
        {"year": {"$eq": 2023}},
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": pre_filter},
)

pre_filtered_documents = pre_filtered_retriever.invoke("How was the reward model trained?")
display_documents(pre_filtered_documents, title="Pre-filtering Results")

In [ ]:
# POST-FILTERING: search first (no filter), then filter in Python
unfiltered_candidates = vector_store.similarity_search(query="How was the reward model trained?", k=15)

post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

print("Documents before post-filtering:", len(unfiltered_candidates))
print("Documents after post-filtering: ", len(post_filtered_documents))

display_documents(post_filtered_documents, title="Post-filtering Results", max_documents=5)

---
## Topic 14: Sparse Retrieval (BM25)

**What:** Classic **keyword search**. It does not understand meaning; it matches the actual words.

**Why "sparse":** each text is represented by a huge vector of all vocabulary words, where most values are 0 (only the words present are non-zero).

**How BM25 scores a chunk:**
- **Term frequency** — the query word appears many times in the chunk → higher score (but with diminishing returns).
- **Rare words matter more (IDF)** — a rare word like "GQA" is more valuable than a common word like "the".
- **Length normalization** — long chunks don't win just because they are long.

**Strong at:** exact terms, abbreviations, names, codes, numbers (`GQA`, `70B`, error codes).
**Weak at:** synonyms. "car" will not match "automobile".

BM25 does **not** need embeddings; it builds its index directly from the chunks.

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4  # number of results

sparse_query = "Grouped-Query Attention GQA 70B"
sparse_documents = bm25_retriever.invoke(sparse_query)

display_documents(sparse_documents, title="Sparse Retrieval: BM25 Results")

---
## Topic 15: Dense Retrieval

**What:** **Meaning-based search** using embeddings (what we did in Topics 8–10).

**Why "dense":** every text is a short vector (384 numbers) where almost every value is non-zero.

**Strong at:** understanding paraphrases and synonyms. The query below never says "GQA", but dense search can still find the GQA chunk because the *meaning* (inference scalability) matches.
**Weak at:** exact rare keywords, IDs and numbers.

In [ ]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

dense_query = "How did Meta improve inference scalability for the largest Llama 2 models?"
dense_documents = dense_retriever.invoke(dense_query)

display_documents(dense_documents, title="Dense Retrieval: Vector Search Results")

### Sparse vs Dense — same query
| | Sparse (BM25) | Dense (Embeddings) |
|---|---|---|
| Matches | exact words | meaning |
| Synonyms | ❌ | ✅ |
| Exact codes / abbreviations | ✅ | ⚠️ sometimes misses |
| Needs embedding model | No | Yes |

Each one finds some chunks the other misses. That is why we combine them next.

In [ ]:
comparison_query = "How did grouped-query attention improve Llama 2 inference scalability?"

show_ids(bm25_retriever.invoke(comparison_query), "SPARSE (BM25) RESULTS")
show_ids(dense_retriever.invoke(comparison_query), "DENSE RESULTS")

---
## Topic 16: Hybrid Retrieval (BM25 + Dense + Weighted RRF)

**What:** Run **both** BM25 and dense search, then merge the two result lists into one ranking. You get exact-keyword power **and** meaning power.

**How the merge works — Reciprocal Rank Fusion (RRF):**
RRF only looks at the **rank position** of each chunk, not the raw scores (BM25 scores and cosine scores are on different scales, so they can't be added directly).

$$\text{score}(d) = \sum_{\text{each retriever}} \frac{\text{weight}}{c + \text{rank}(d)}$$

- `rank` = position of the chunk in that retriever's list (1, 2, 3...)
- `c` = 60 (a constant that stops the top rank from dominating)
- `weight` = how much you trust each retriever

A chunk that appears high in **both** lists gets the biggest final score.

`EnsembleRetriever` does all of this for us. `weights=[0.5, 0.5]` → equal trust in BM25 and dense.

```
                    ┌── BM25 Retriever ─────┐
User query ─────────┤                       ├── Weighted RRF ──► Combined ranking
                    └── Dense Retriever ────┘
```

In [ ]:
# Give each retriever a bigger pool so the fusion has more to work with
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8},
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],  # [BM25 weight, Dense weight]
)

hybrid_query = "Llama 2 70B grouped-query attention and inference scalability"
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

display_documents(hybrid_documents, title="Hybrid Retrieval: BM25 + Dense + RRF", max_documents=6)

### Comparing all three on one query

In [ ]:
test_query = "How did Meta make Llama 2 70B efficient for large-scale inference?"

show_ids(bm25_retriever.invoke(test_query)[:4], "1. Sparse Retrieval")
show_ids(dense_retriever.invoke(test_query)[:4], "2. Dense Retrieval")
show_ids(hybrid_retriever.invoke(test_query)[:4], "3. Hybrid Retrieval")

---
## Topic 17: Groq LLM Setup

The next techniques (query rewriting, expansion, decomposition, HyDE) use an **LLM to improve the query before searching**. We use Groq because it is very fast.

- **Model:** `openai/gpt-oss-120b` (Groq's recommended production model; the older `llama-3.3-70b-versatile` was shut down in August 2026).
- **`temperature=0`** → same input gives (almost) the same output. Good for search tasks.
- You can change the model with the `GROQ_CHAT_MODEL` environment variable. `openai/gpt-oss-20b` is a smaller, cheaper option.

In [ ]:
CHAT_MODEL = os.environ.get("GROQ_CHAT_MODEL", "openai/gpt-oss-120b")

llm = ChatGroq(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)
print("Test reply:", llm.invoke("Reply with one word: ready").content)

---
## Topic 18: Query Rewriting

**Problem:** In a chat, users ask follow-up questions like *"What did Meta do after that?"*. The word **"that"** means nothing to a retriever, so search fails.

**Solution:** Give the LLM the conversation history and ask it to **rewrite the question into a complete, standalone search query**.

```
"What did Meta do after that?"
            ↓  (LLM + chat history)
"What steps did Meta take after the supervised fine-tuning of Llama 2-Chat?"
```

The chain is: `prompt | llm | StrOutputParser()` → the parser returns plain text instead of a message object.

In [ ]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''You rewrite conversational questions into clear, standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.''',
        ),
        (
            "human",
            '''Conversation history:
{chat_history}

Current query:
{query}''',
        ),
    ]
)

query_rewriting_chain = query_rewriting_prompt | llm | StrOutputParser()

In [ ]:
chat_history = '''
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
'''

original_query = "What did Meta do after that?"

rewritten_query = query_rewriting_chain.invoke(
    {"chat_history": chat_history, "query": original_query}
).strip()

print("Original query: ", original_query)
print("Rewritten query:", rewritten_query)

rewritten_query_documents = hybrid_retriever.invoke(rewritten_query)
display_documents(rewritten_query_documents, title="Documents Retrieved Using the Rewritten Query", max_documents=5)

---
## Topic 19: Query Expansion

**Problem:** The user's words may not match the words used in the document (e.g. user says "human feedback", paper says "RLHF" or "preference data").

**Solution:** Ask the LLM to generate **several versions of the same question** (synonyms, abbreviations, technical terms). Search with **all** of them, combine the results and remove duplicates. This increases **recall** (finding more of the relevant chunks).

**Structured output:** `llm.with_structured_output(ExpandedQueryOutput)` forces the LLM to return a clean Python object (a list of queries) instead of free text, so we don't have to parse it ourselves.

In [ ]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description="Four alternative search queries expressing the same information need using different wording."
    )


query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.''',
        ),
        ("human", "Original query: {query}"),
    ]
)

query_expansion_chain = query_expansion_prompt | llm.with_structured_output(ExpandedQueryOutput)

Helper to remove duplicate chunks (the same chunk is often found by several queries). It keeps the first occurrence and preserves order.

In [ ]:
def deduplicate_documents(documents):
    # Remove duplicate chunks while keeping their order.
    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = document.metadata.get("chunk_id") or (
            document.metadata.get("source"),
            document.metadata.get("page"),
            document.page_content,
        )
        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [ ]:
original_query = "How was Llama 2-Chat improved using human feedback?"

expanded_output = query_expansion_chain.invoke({"query": original_query})
all_expanded_queries = [original_query, *expanded_output.queries]

print("Generated search queries:")
for number, q in enumerate(all_expanded_queries, start=1):
    print(f"{number}. {q}")

# Search with every query, then merge and deduplicate
expanded_query_documents = []
for q in all_expanded_queries:
    expanded_query_documents.extend(dense_retriever.invoke(q))

expanded_query_documents = deduplicate_documents(expanded_query_documents)

display_documents(expanded_query_documents, title="Query Expansion: Combined Unique Documents", max_documents=10)

---
## Topic 20: MultiQueryRetriever (Built-in Query Expansion)

LangChain already has query expansion built in. `MultiQueryRetriever`:
1. asks the LLM for several versions of the question,
2. runs the base retriever for each,
3. returns the **unique union** of all results.

`include_original=True` → also searches with the user's original question.

It is the same idea as Topic 19, but in just a few lines.

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

multi_query_documents = multi_query_retriever.invoke("How did human feedback improve Llama 2-Chat?")
display_documents(multi_query_documents, title="Built-in MultiQueryRetriever Results", max_documents=10)

---
## Topic 21: Query Decomposition

**Problem:** A complex question asks **several things at once**, e.g. *"Compare pretraining with fine-tuning, and explain safety"*. One search can't cover all parts well.

**Solution:** The LLM **breaks the question into small, independent (atomic) sub-questions**. We search for each one separately and merge the evidence.

**Expansion vs Decomposition:**
- **Expansion** → *same* question, different words.
- **Decomposition** → *different* smaller questions that together answer the big one.

```
Complex query
     ↓
Atomic sub-queries
     ↓
Retrieve for every sub-query
     ↓
Merge evidence → Remove duplicates
```

In [ ]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description="Independent and atomic search queries required to answer the complete user question."
    )


query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Break the user's complex question into independent, atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.''',
        ),
        ("human", "Complex query: {query}"),
    ]
)

query_decomposition_chain = query_decomposition_prompt | llm.with_structured_output(DecomposedQueryOutput)

In [ ]:
complex_query = "Compare Llama 2 pretraining with Llama 2-Chat fine-tuning, and explain how Meta improved model safety."

sub_queries = query_decomposition_chain.invoke({"query": complex_query}).sub_queries

print("Original complex query:\n", complex_query)
print("\nGenerated sub-queries:")
for number, q in enumerate(sub_queries, start=1):
    print(f"{number}. {q}")

# Retrieve for each sub-query
decomposition_results = {q: hybrid_retriever.invoke(q) for q in sub_queries}

for sub_query, documents in decomposition_results.items():
    show_ids(documents[:4], f"Sub-query: {sub_query}")

In [ ]:
# Merge evidence from all sub-queries and remove duplicates
all_decomposition_documents = []
for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(all_decomposition_documents)

display_documents(all_decomposition_documents, title="Combined Evidence from All Sub-Queries", max_documents=12)

---
## Topic 22: HyDE — Hypothetical Document Embeddings (Custom Retriever)

**Problem:** A short question and a long answer paragraph look different, so their embeddings may not be very close.
- Question: *"How did Meta use human feedback?"*
- Paper text: *"Annotators selected the preferred of two responses... a reward model was trained... PPO..."*

**Idea:** Ask the LLM to **write a fake (hypothetical) answer paragraph** first. A fake answer *looks like* the real paper text, so its embedding lands closer to the real chunks. Then we search with the fake answer's embedding.

> The fake passage may contain wrong facts — that is fine. It is **only used for searching**. The final answer must always come from the **real retrieved documents**.

```
User query
     ↓
LLM writes a hypothetical answer passage
     ↓
Embed the hypothetical passage
     ↓
Dense vector search
     ↓
Real source documents
```

### Custom retriever
LangChain gives a `BaseRetriever` class. To build your own retriever, inherit from it and write your logic inside `_get_relevant_documents()`. Then it works with `.invoke()` like every other retriever.

In [ ]:
class HyDERetriever(BaseRetriever):
    # Generate a hypothetical passage, then retrieve REAL documents with it.

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        # Step 1: LLM writes a fake answer passage
        hypothetical_document = self.hypothesis_chain.invoke({"query": query}).strip()
        self.last_hypothetical_document = hypothetical_document

        # Step 2: search the vector store with that passage
        return self.vector_retriever.invoke(
            hypothetical_document,
            config={"callbacks": run_manager.get_child()},
        )

In [ ]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.''',
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = hyde_prompt | llm | StrOutputParser()

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5}),
)

In [ ]:
hyde_query = "How did Meta use human feedback to improve the helpfulness and safety of Llama 2-Chat?"

hyde_documents = hyde_retriever.invoke(hyde_query)

print("Original query:\n", hyde_query)
print("\nHypothetical document used for retrieval:\n", hyde_retriever.last_hypothetical_document)

display_documents(hyde_documents, title="HyDE Retrieval Results", max_documents=5)

---
## Topic 23: HyDE with LangChain's Built-in Helper

`HypotheticalDocumentEmbedder` is **not a retriever** — it is an **embeddings wrapper**. When you embed a query with it, it first asks the LLM for a hypothetical document and then embeds that document.

So we open the **same Chroma collection** but pass the HyDE wrapper as the embedding function. Now every query is automatically HyDE-transformed before searching.

> **Note:** Never use the HyDE wrapper when *building* the index. Real chunks must be embedded with the normal `embeddings` object (Topic 6). The wrapper is used **only for queries**.

In [ ]:
built_in_hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key="web_search",  # one of LangChain's ready-made HyDE prompts
)

built_in_hyde_vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=built_in_hyde_embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = built_in_hyde_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

built_in_hyde_documents = built_in_hyde_retriever.invoke(
    "How did Meta use human feedback to improve the helpfulness and safety of Llama 2-Chat?"
)

display_documents(built_in_hyde_documents, title="Built-in LangChain HyDE Results", max_documents=5)

---
## Topic 24: Reranking with a Cross-Encoder

**Two-stage retrieval:**
1. **Stage 1 (fast, broad):** get many candidates, e.g. top **20**, with vector search.
2. **Stage 2 (slow, precise):** a **reranker** reads each (query, chunk) pair carefully and re-orders them. Keep only the best **5**.

**Bi-encoder vs Cross-encoder:**
| | **Bi-encoder** (embeddings) | **Cross-encoder** (reranker) |
|---|---|---|
| How | query and chunk are embedded **separately**, then compared | query and chunk are read **together** in one model |
| Speed | very fast (chunk vectors are pre-computed) | slow (must run for every pair) |
| Accuracy | good | **better**, it sees how query words relate to chunk words |
| Used for | searching millions of chunks | re-ordering a small candidate list |

That's why we use the fast one first and the accurate one second.

**Model:** `cross-encoder/ms-marco-MiniLM-L6-v2` (free, runs locally on CPU).

**`ContextualCompressionRetriever`** = base retriever + a "compressor" that post-processes results. Here the compressor is the reranker, which keeps only the `top_n=5` best chunks.

```
User query → Dense retriever → Top 20 candidates → Cross-Encoder reranker → Final top 5
```

In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20},
)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
reranking_query = "How did Meta collect and use human preference data to train Llama 2-Chat?"

initial_candidates = candidate_retriever.invoke(reranking_query)
show_ids(initial_candidates[:10], "BEFORE reranking (vector search order, top 10 of 20):")

reranked_documents = reranking_retriever.invoke(reranking_query)
display_documents(reranked_documents, title="AFTER reranking: Final Top 5", max_documents=5)

---
## Topic 25: Hybrid Retrieval + Reranking (Best Pipeline)

This combines the best of everything:
- **BM25** catches exact keywords,
- **Dense** catches meaning,
- **RRF** merges them into one candidate list,
- **Cross-encoder** picks the truly best chunks.

```
                    ┌── BM25 Search ───────┐
User query ─────────┤                      ├── Weighted RRF
                    └── Dense Search ──────┘
                                              ↓
                                     Candidate documents
                                              ↓
                                  Cross-Encoder Reranker
                                              ↓
                                     Final top documents
```

This "hybrid + rerank" setup is very common in production RAG systems.

In [ ]:
bm25_retriever.k = 15

dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15},
)

hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_candidate_retriever],
    weights=[0.5, 0.5],
)

hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

query = "What techniques did Meta use to improve the helpfulness and safety of Llama 2-Chat?"
final_documents = hybrid_reranking_retriever.invoke(query)

display_documents(final_documents, title="Hybrid Retrieval + Cross-Encoder Reranking", max_documents=5)

---
## Topic 26: Summary of All Techniques

| Technique | One-line idea | Best for |
|---|---|---|
| **Similarity search** | top-k closest vectors | simple, default search |
| **MMR** | relevant **and** diverse | avoiding repeated chunks |
| **Score threshold** | only results above a minimum score | avoiding junk context |
| **Metadata filtering** | search only matching chunks | sections, users, dates, permissions |
| **Sparse (BM25)** | keyword matching | exact terms, codes, abbreviations |
| **Dense** | meaning matching | paraphrases, synonyms |
| **Hybrid (RRF)** | BM25 + Dense merged by rank | best general recall |
| **Query rewriting** | turn a follow-up into a standalone query | chatbots with history |
| **Query expansion / MultiQuery** | many versions of one question | vocabulary mismatch |
| **Query decomposition** | split into smaller questions | complex multi-part questions |
| **HyDE** | search with a fake answer passage | short questions vs long documents |
| **Reranking** | cross-encoder re-orders candidates | precision of the final top results |
| **Hybrid + Reranking** | all of the above together | production RAG |

---
## Topic 27: Homework Topics

Study these next. A short idea for each:

| Topic | Short idea |
|---|---|
| **Weighted fusion** | Combine scores from several retrievers using weights (e.g. 0.7 dense + 0.3 BM25) after normalizing their scores. |
| **Reciprocal Rank Fusion (RRF)** | Merge ranked lists using `weight / (60 + rank)`; only ranks matter, not raw scores (used in Topic 16). |
| **MultiQueryRetriever** | LLM creates several versions of the query and the results are merged (Topic 20). |
| **Multi-hop retriever** | Answer needs facts from several steps: search → read → use what you found to make the next search. |
| **Parent document retriever** | Search with small chunks (precise), but return the bigger parent chunk/page (more context). |
| **Sentence window retriever** | Search single sentences, then return the sentence plus a few sentences before and after it. |
| **Contextual compression** | After retrieval, cut out irrelevant parts or drop weak chunks before sending them to the LLM (the reranker in Topic 24 is one type). |
| **Custom retriever** | Inherit `BaseRetriever` and write your own `_get_relevant_documents()` logic (like `HyDERetriever` in Topic 22). Works in LangChain and LangGraph. |